In [1095]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

In [1096]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

url = "https://www.flipkart.com/j9e/abm/c54/~cs-npe3jninbr/pr?sid=j9e,abm,c54&collection-tab-name=1.5+Ton+ACs&param=8334&BU=Appliances"
response = requests.get(url,headers=headers)
webpage = response.text

In [1097]:
soup = BeautifulSoup(webpage,'lxml')

In [1098]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <link href="https://rukminim2.flixcart.com" rel="preconnect"/>
  <link href="//static-assets-web.flixcart.com/fk-p-linchpin-web/fk-cp-zion/css/bundle.aa1465.css" rel="stylesheet"/>
  <link href="//static-assets-web.flixcart.com/fk-p-linchpin-web/fk-cp-zion/css/bundle.fbaef9.css" rel="stylesheet"/>
  <meta content="text/html; charset=utf-8" http-equiv="Content-type"/>
  <meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
  <meta content="102988293558" property="fb:page_id"/>
  <meta content="658873552,624500995,100000233612389" property="fb:admins"/>
  <link href="https://static-assets-web.flixcart.com/www/promos/new/20150528-140547-favicon-retina.ico" rel="shortcut icon"/>
  <link href="/osdd.xml?v=2" rel="search" type="application/opensearchdescription+xml"/>
  <meta content="website" property="og:type"/>
  <meta content="Flipkart.com" name="og_site_name" property="og:site_name"/>
  <link href="/apple-touch-icon-57x57.png" rel="apple-touch

In [1099]:
appliances = soup.find_all('div',class_='jIjQ8S')

In [1100]:
name = []
rating = []
reviews = []
price = []
discount = []
warranty = []

for i in appliances:
    name.append(i.find('div',class_='RG5Slk').get_text(strip=True))

    rating.append(i.find('div',class_='MKiFS6').get_text(strip=True))

    spans = i.find_all('span')

    for s in spans:
        text = s.get_text(strip=True)
        if 'Reviews' in text:
            reviews.append(text)
            break
    else:
        reviews.append(None)

    price.append(i.find('div', class_='hZ3P6w DeU9vF').get_text(strip=True))
    discount.append(i.find('div',class_='HQe8jr'))
    warranty.append(i.find_all('li',class_='DTBslk'))

In [1101]:
d = {'Name':name, 'Rating':rating, 'Reviews': reviews, 'Price':price, 'Warranty': warranty, 'Discount':discount,}

In [1102]:
df = pd.DataFrame(d)

In [1103]:
df.head(2)

,Name,Rating,Reviews,Price,Warranty,Discount
0,MarQ by Flipkart 2025 Model 1.5 Ton 3 Star Spl...,4.2,"49,542 Ratings&6,291 Reviews","₹24,990","[[Annual Power Usage: 953.67 Units], [Room Siz...",[[47% off]]
1,Voltas 2025 Model 1.5 Ton 3 Star Split Inverte...,4.2,"81,429 Ratings&6,881 Reviews","₹31,490","[[Annual Power Usage: 975.26 Units], [Room Siz...",[[45% off]]


In [1104]:
def extract_features(text):
    if not text:
        return None

    text = text.strip()
    data = {}

    # Brand
    data['brand'] = text.split()[0]

    # Year
    year = re.search(r'\b(20\d{2})\b', text)
    data['model_year'] = year.group() if year else None

    # Ton
    ton = re.search(r'(\d+(\.\d+)?)\s*Ton', text)
    data['ton'] = ton.group().split()[0] if ton else None

    # Star rating
    star = re.search(r'(\d+)\s*Star', text)
    data['star'] = star.group().split()[0] if star else None

    # Type
    data['type'] = 'Split' if 'Split' in text else None

    # Inverter
    data['inverter'] = 'Inverter' if 'Inverter' in text else None

    return data



Brand = []
Model_year = []
Ton = []
Star = []

for n in name:
    data = extract_features(n)
    
    Brand.append(data['brand'])
    Model_year.append(data['model_year'])
    Ton.append(data['ton'])
    Star.append(data['star'])

In [1105]:
df['Name'].apply(extract_features)

0     {'brand': 'MarQ', 'model_year': '2025', 'ton':...
1     {'brand': 'Voltas', 'model_year': '2025', 'ton...
2     {'brand': 'Voltas', 'model_year': '2025', 'ton...
3     {'brand': 'MarQ', 'model_year': '2025', 'ton':...
4     {'brand': 'MarQ', 'model_year': '2025', 'ton':...
5     {'brand': 'IFB', 'model_year': '2025', 'ton': ...
6     {'brand': 'Voltas', 'model_year': '2025', 'ton...
7     {'brand': 'Godrej', 'model_year': '2025', 'ton...
8     {'brand': 'Voltas', 'model_year': '2025', 'ton...
9     {'brand': 'Sansui', 'model_year': '2025', 'ton...
10    {'brand': 'Panasonic', 'model_year': '2025', '...
11    {'brand': 'LG', 'model_year': '2025', 'ton': '...
12    {'brand': 'Lloyd', 'model_year': '2025', 'ton'...
13    {'brand': 'Blue', 'model_year': '2025', 'ton':...
14    {'brand': 'LG', 'model_year': '2025', 'ton': '...
15    {'brand': 'Godrej', 'model_year': '2025', 'ton...
16    {'brand': 'Lloyd', 'model_year': '2025', 'ton'...
17    {'brand': 'CARRIER', 'model_year': '2024',

In [1106]:
d = {'Brand':Brand, 'Model_Year':Model_year, 'Ton':Ton, 'Star':Star, 'Rating':rating, 'Reviews': reviews, 'Price':price, 'Warranty': warranty, 'Discount':discount,}

In [1107]:
df = pd.DataFrame(d)

In [1108]:
df.sample(5)

,Brand,Model_Year,Ton,Star,Rating,Reviews,Price,Warranty,Discount
22,Panasonic,2025,1.5,3,4.2,"23,768 Ratings&2,315 Reviews","₹33,490","[[Annual Power Usage: 977.16 Units], [Room Siz...",[[35% off]]
9,Sansui,2025,1.5,5,4.1,"10,647 Ratings&2,262 Reviews","₹28,790","[[Annual Power Usage: 767.34 W], [Room Size: 1...",[[55% off]]
19,Godrej,2025,1.5,5,4,"11,969 Ratings&1,155 Reviews","₹33,991","[[Annual Power Usage: 728.31 Units], [Room Siz...",[[35% off]]
1,Voltas,2025,1.5,3,4.2,"81,429 Ratings&6,881 Reviews","₹31,490","[[Annual Power Usage: 975.26 Units], [Room Siz...",[[45% off]]
21,Haier,2025,1.5,5,4,"2,926 Ratings&308 Reviews","₹45,990","[[Annual Power Usage: 836.95 kWh], [Room Size:...",[[47% off]]


In [1109]:
df['Warranty'] = df['Warranty'].apply(
    lambda x: x[2] if isinstance(x, list) and len(x) > 0 else None
)

In [1110]:
df.sample(5)

,Brand,Model_Year,Ton,Star,Rating,Reviews,Price,Warranty,Discount
7,Godrej,2025,1.5,4,3.9,"10,667 Ratings&776 Reviews","₹30,490",[5 Years Warranty on Whole Product (Valid on s...,[[33% off]]
10,Panasonic,2025,1.5,5,4.3,"21,459 Ratings&2,269 Reviews","₹39,990","[1 Year Comprehensive warranty on Product,\n5 ...",[[32% off]]
6,Voltas,2025,1.5,3,4.2,"81,429 Ratings&6,881 Reviews","₹31,490",[5 years warranty from the date of installatio...,[[45% off]]
1,Voltas,2025,1.5,3,4.2,"81,429 Ratings&6,881 Reviews","₹31,490",[5 years warranty from the date of installatio...,[[45% off]]
19,Godrej,2025,1.5,5,4,"11,969 Ratings&1,155 Reviews","₹33,991",[5 Years Warranty on Whole Product (Valid on s...,[[35% off]]


In [1111]:
df['Reviews'] = df['Reviews'].str.extract(r'&([\d,]+)\s*Reviews')[0]

In [1112]:
df.sample(5)

,Brand,Model_Year,Ton,Star,Rating,Reviews,Price,Warranty,Discount
13,Blue,2025,1.5,5,4.1,"1,179","₹40,990",[60 months for those customers who purchase Bl...,[[40% off]]
3,MarQ,2025,1.5,5,4.2,"3,576","₹28,990",[1 Year Warranty on Product and 10 Years Warra...,[[48% off]]
2,Voltas,2025,1.5,5,4.2,"3,763","₹38,990",[5 years warranty from the date of installatio...,[[44% off]]
12,Lloyd,2025,1.5,3,4.1,"2,145","₹30,990","[1 Year Warranty on Product, 5 Year Warranty o...",[[42% off]]
4,MarQ,2025,1.5,3,4.2,"6,291","₹24,990",[1 Year Warranty on Product and 10 Years Warra...,[[47% off]]


In [1113]:
# --- Reviews ---
df['Reviews'] = (
    df['Reviews']
    .astype(str)
    .str.replace(',', '', regex=False)
    .astype(int)
)

# --- Price ---
df['Price'] = (
    df['Price']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(int)
)

# --- Warranty (list → string) ---
df['Warranty'] = df['Warranty'].apply(
    lambda x: " | ".join(x) if isinstance(x, list) else x
)

# --- Discount (extract percentage safely) ---
def extract_discount(x):
    match = re.search(r'(\d+)%', str(x))
    return int(match.group(1)) if match else None

df['Discount'] = df['Discount'].apply(extract_discount)

In [1114]:
df.sample(5)

,Brand,Model_Year,Ton,Star,Rating,Reviews,Price,Warranty,Discount
1,Voltas,2025,1.5,3,4.2,6881,31490,[5 years warranty from the date of installatio...,45
13,Blue,2025,1.5,5,4.1,1179,40990,[60 months for those customers who purchase Bl...,40
2,Voltas,2025,1.5,5,4.2,3763,38990,[5 years warranty from the date of installatio...,44
23,Daikin,2025,1.5,5,4.3,1509,42490,[1 Year Warranty on Product and 5 Years Warra...,31
11,LG,2025,1.5,5,4.2,4192,42990,"[1 Year Comprehensive Warranty on Product, 5 Y...",45


In [1115]:
final_df = pd.DataFrame()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

for page in range(1, 5):

    url = f"https://www.flipkart.com/j9e/abm/c54/~cs-npe3jninbr/pr?sid=j9e,abm,c54&collection-tab-name=1.5+Ton+ACs&page={page}"
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'lxml')

    appliances = soup.find_all('div', class_='jIjQ8S')

    data = {
        "Name": [],
        "Rating": [],
        "Reviews": [],
        "Price": [],
        "Warranty": [],
        "Discount": []
    }

    for i in appliances:

        # Name
        name_tag = i.find('div', class_='RG5Slk')
        data["Name"].append(name_tag.get_text(strip=True) if name_tag else None)

        # Rating
        rating_tag = i.find('div', class_='MKiFS6')
        data["Rating"].append(rating_tag.get_text(strip=True) if rating_tag else None)

        # Reviews
        spans = i.find_all('span')
        review_text = None
        for s in spans:
            text = s.get_text(strip=True)
            if 'Reviews' in text:
                review_text = text
                break
        data["Reviews"].append(review_text)

        # Price
        price_tag = i.find('div', class_='hZ3P6w DeU9vF')
        data["Price"].append(price_tag.get_text(strip=True) if price_tag else None)

        # Discount
        discount_tag = i.find('div', class_='HQe8jr')
        data["Discount"].append(discount_tag.get_text(strip=True) if discount_tag else None)

        # Warranty
        warranty_tag = i.find_all('li', class_='DTBslk')
        data["Warranty"].append([w.get_text(strip=True) for w in warranty_tag])

    df = pd.DataFrame(data)

    # ---------------- CLEANING ----------------

    df['Reviews'] = df['Reviews'].str.extract(r'&([\d,]+)\s*Reviews')[0]
    df['Reviews'] = df['Reviews'].str.replace(',', '').astype(float)

    df['Price'] = df['Price'].str.replace('₹', '', regex=False).str.replace(',', '', regex=False)
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

    df['Warranty'] = df['Warranty'].apply(
        lambda x: " | ".join(x) if isinstance(x, list) else x
    )

    df['Discount'] = df['Discount'].apply(
        lambda x: int(re.search(r'(\d+)%', str(x)).group(1)) if re.search(r'(\d+)%', str(x)) else None
    )

    # Append page data
    final_df = pd.concat([final_df, df], ignore_index=True)


In [1116]:
final_df.sample(5)

,Name,Rating,Reviews,Price,Warranty,Discount
14,LG 2025 Model 1.5 Ton 3 Star Split AI Dual Inv...,4,3922.0,33990,Annual Power Usage: 852.44 Units | Room Size: ...,53.0
73,LG 2025 Model 1.5 Ton 3 Star Window Dual Inver...,4.1,445.0,37990,Annual Power Usage: 1115.04 Units | Room Size:...,48.0
44,Sharp 2025 Model 1.5 Ton 5 Star Split Inverter...,3.8,4.0,34590,Annual Power Usage: 782.92 kWh | Room Size: 11...,46.0
7,Godrej 2025 Model 1.5 Ton 4 Star Split Inverte...,3.9,776.0,30490,Annual Power Usage: 835.07 Units | Room Size: ...,33.0
51,Voltas 2025 Model 1.5 Ton 3 Star Split Inverte...,4.2,6881.0,31490,Annual Power Usage: 975.26 Units | Room Size: ...,45.0


In [1117]:
final_df['Warranty'][0]

'Annual Power Usage: 953.67 Units | Room Size: 111 - 150 sqft | 1 Year Warranty on Product and 10 Years Warranty on Compressor'

In [1118]:
import numpy as np

# ---- UNITS (flexible for Units / kWh / numbers) ----
final_df['Units'] = final_df['Warranty'].str.extract(r'([\d.]+)\s*(Units|kWh)?')[0]
final_df['Units'] = pd.to_numeric(final_df['Units'], errors='coerce')

# ---- ROOM SIZE (any format like 111 - 150) ----
final_df['Room_Size'] = final_df['Warranty'].str.extract(r'(\d+\s*-\s*\d+)')[0]

# ---- WARRANTY (grab first meaningful match) ----
final_df['Warranty'] = final_df['Warranty'].apply(
    lambda x: re.search(r'(\d+\s*(?:Year|Yr)s?.*?Product)', str(x), re.IGNORECASE)
)

final_df['Warranty'] = final_df['Warranty'].apply(
    lambda x: x.group(1) if x else None
)

In [1121]:
final_df.sample(4)

,Name,Rating,Reviews,Price,Warranty,Discount,Units,Room_Size
37,Haier 2025 Model 1.5 Ton 5 Star Split Inverter...,4,308.0,37399,5 years on the product,47.0,714.29,111 - 150
28,Blue Star 2025 Model 1.3 Ton 3 Star Split Inve...,4.1,140.0,32400,1 Year Warranty on Product,43.0,865.48,111 - 150
67,Haier 2025 Model 1.5 Ton 3 Star Split Dual Inv...,4,568.0,36990,5 years on the product,43.0,1029.15,111 - 150
50,MarQ by Flipkart 2025 Model 1.5 Ton 3 Star Spl...,4.2,6291.0,24990,1 Year Warranty on Product,47.0,953.67,111 - 150


In [ ]:
final_df.to_csv('Air_Conditioner.csv')